In [4]:
# ==============================================================================
# 🐍 AFRICAN SNAKEBITE SPECIES IDENTIFICATION — PRODUCTION PIPELINE
# ==============================================================================
# This script consolidates your prototypes into an end-to-end framework:
# 1. Downloads datasets and filters specifically for 6 African medical targets.
# 2. Automatically reorganizes messy dataset sub-folders into standard layouts.
# 3. Trains a partially fine-tuned EfficientNet-B0 with differential learning rates.
# 4. Injects a clinical lookup tier (venom action and risk) onto inference vectors.
# ==============================================================================

# --- STEP 0: SETUP ENVIRONMENT & DEPENDENCIES ---
# Install all required packages at the beginning to ensure they are available for imports.
# 'splitfolders' is installed as 'split-folders'.
!pip install split-folders kagglehub scikit-learn --quiet

import os
import json
import random
import shutil
import tarfile
import splitfolders
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, models, transforms
from PIL import Image
import matplotlib.pyplot as plt
import kagglehub # Added to general imports

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Running execution pipeline on device: {DEVICE}")

# --- STEP 1: CONFIGURE SYSTEM CONSTANTS ---
MAX_IMAGES_PER_SPECIES = 800
CLEAN_DATASET_DIR = "dataset_clean"
SPLIT_DIR = "dataset_split"
MODEL_SAVE_PATH = "african_snake_efficientnet.pth"

AFRICAN_TARGETS = [
    "Bitis_arietans",          # Puff Adder
    "Echis_ocellatus",         # West African Carpet Viper
    "Bitis_gabonica",          # Gaboon Viper
    "Dendroaspis_polylepis",   # Black Mamba
    "Naja_nigricollis",        # Black-necked Spitting Cobra
    "Dispholidus_typus",       # Boomslang
]

# --- STEP 2: DOWNLOAD SOURCE IMAGE BANK & METADATA ---
print("\n--- Downloading Data Archives via Hub Vectors ---")
# Geographic distribution + species metadata (cross-checking common names)
geo_path = kagglehub.dataset_download("saidmassinissafazez/snakes-species-details-dataset")
print(f"Geographic metadata located at: {geo_path}")

# Ingesting the core SnakeCLEF archive
tar_path = "SnakeCLEF2023-train-medium_size.tar.gz"
if not os.path.exists(tar_path):
    print("Downloading SnakeCLEF2023 image archive (~17GB). This might take several minutes...")
    !wget --no-check-certificate https://ptak.felk.cvut.cz/plants/plants/SnakeCLEF2023/SnakeCLEF2023-train-medium_size.tar.gz
else:
    print("Target image archive tarball already exists locally. Proceeding to parsing index.")

# --- STEP 3: NON-DESTRUCTIVE SCAN & EXTRACTION ---
print("\n--- Scanning Tar Index for Target Filtering ---")
target_files = []
extracted_paths = {species: [] for species in AFRICAN_TARGETS}

with tarfile.open(tar_path, "r:gz") as tar:
    for member in tar:
        if not member.isfile():
            continue
        path = member.name
        for species in AFRICAN_TARGETS:
            if species in path or species.replace("_", " ") in path:
                extracted_paths[species].append(path)
                break

# Balance datasets explicitly via constrained cap tracking
for species, paths in extracted_paths.items():
    sampled = random.sample(paths, MAX_IMAGES_PER_SPECIES) if len(paths) > MAX_IMAGES_PER_SPECIES else paths
    target_files.extend(sampled)
    print(f"  {species.replace('_', ' '):28s}: Selected {len(sampled)} items.")

with open("target_files.txt", "w") as f:
    f.write("\n".join(target_files))

print("Extracting isolated target files from archive...")
!tar -xzf {tar_path} -T target_files.txt
print("Target extraction complete.")

# --- STEP 4: FLATTEN ROOT DIRECTORY TREE LAYOUTS ---
print("\n--- Flattening and Cleaning Directory Layouts ---")
EXTRACTED_ROOT = "SnakeCLEF2023-medium_size"

for s in AFRICAN_TARGETS:
    os.makedirs(os.path.join(CLEAN_DATASET_DIR, s), exist_ok=True)

copied_count = 0
if os.path.exists(EXTRACTED_ROOT):
    for year in os.listdir(EXTRACTED_ROOT):
        year_path = os.path.join(EXTRACTED_ROOT, year)
        if not os.path.isdir(year_path):
            continue
        for species in AFRICAN_TARGETS:
            species_src = os.path.join(year_path, species)
            if not os.path.isdir(species_src):
                continue
            for img in os.listdir(species_src):
                src = os.path.join(species_src, img)
                dst = os.path.join(CLEAN_DATASET_DIR, species, f"{year}_{img}")
                if not os.path.exists(dst):
                    shutil.copy2(src, dst)
                    copied_count += 1
print(f"Dataset flattened successfully. Unified {copied_count} total assets.")

# Split folder directories automatically into standardized partitions (70/15/15)
splitfolders.ratio(CLEAN_DATASET_DIR, output=SPLIT_DIR, seed=42, ratio=(0.7, 0.15, 0.15))

# --- STEP 5: CLINICAL LOOKUP DEFINITIONS ---
VENOM_REFERENCE = {
    "Bitis_arietans": {
        "common_name": "Puff Adder",
        "fang_type": "Solenoglyphous (front, hinged, hollow)",
        "venom_risk": "High",
        "venom_action": "Cytotoxic / tissue-damaging",
        "clinical_note": "Responsible for high fatality counts; causes severe local swelling and tissue necrosis."
    },
    "Echis_ocellatus": {
        "common_name": "West African Carpet Viper",
        "fang_type": "Solenoglyphous (front, hinged, hollow)",
        "venom_risk": "High",
        "venom_action": "Haemotoxic / anticoagulant",
        "clinical_note": "A primary source of severe systemic hemorrhage and coagulopathy across West Africa."
    },
    "Bitis_gabonica": {
        "common_name": "Gaboon Viper",
        "fang_type": "Solenoglyphous (front, hinged, hollow)",
        "venom_risk": "High",
        "venom_action": "Cytotoxic and haemotoxic",
        "clinical_note": "Possesses the longest fangs of any venomous snake species along with large venom yields."
    },
    "Dendroaspis_polylepis": {
        "common_name": "Black Mamba",
        "fang_type": "Proteroglyphous (front, fixed)",
        "venom_risk": "Very high",
        "venom_action": "Neurotoxic",
        "clinical_note": "Rapidly causes neuromuscular blockages; demands rapid antivenom therapeutic deployment."
    },
    "Naja_nigricollis": {
        "common_name": "Black-necked Spitting Cobra",
        "fang_type": "Proteroglyphous (front, fixed)",
        "venom_risk": "High",
        "venom_action": "Cytotoxic; capable of defensive airborne venom projection",
        "clinical_note": "Ocular exposure induces rapid chemical conjunctivitis and requires immediate irrigation."
    },
    "Dispholidus_typus": {
        "common_name": "Boomslang",
        "fang_type": "Opisthoglyphous (rear, grooved)",
        "venom_risk": "High",
        "venom_action": "Haemotoxic (Slow-acting)",
        "clinical_note": "Symptom presentation can be delayed for hours, creating a risk of hidden internal bleeding."
    }
}

# --- STEP 6: TRANSFORMATIONS & DATA LOADERS ---
data_transforms = {
    "train": transforms.Compose([
        transforms.RandomResizedCrop(224),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(15),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
    "eval": transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ]),
}

train_set = datasets.ImageFolder(os.path.join(SPLIT_DIR, "train"), transform=data_transforms["train"])
val_set = datasets.ImageFolder(os.path.join(SPLIT_DIR, "val"), transform=data_transforms["eval"])

CLASS_NAMES = train_set.classes
train_loader = DataLoader(train_set, batch_size=16, shuffle=True, num_workers=2)
val_loader = DataLoader(val_set, batch_size=16, shuffle=False, num_workers=2)

# --- STEP 7: PARTIAL FINE-TUNING & DESIGN MODEL HEAD ---
print("\n--- Constructing Partial Fine-Tuning Architectural Parameters ---")
model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)

# Freeze parameters universally
for param in model.parameters():
    param.requires_grad = False

# Selectively unfreeze the last 2 feature blocks to optimize fine skin patterns
for block in model.features[-2:]:
    for param in block.parameters():
        param.requires_grad = True

# Re-engineer classification boundary heads
num_features = model.classifier[1].in_features
model.classifier[1] = nn.Linear(num_features, len(CLASS_NAMES))
model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss()

# Configure differential learning rates: backbone parameters adapt slower than the head
optimizer = optim.AdamW([
    {"params": model.features.parameters(), "lr": 1e-4},
    {"params": model.classifier.parameters(), "lr": 1e-3}
], weight_decay=1e-4)



Running execution pipeline on device: cpu

--- Downloading Data Archives via Hub Vectors ---


100%|██████████| 75.4k/75.4k [00:00<00:00, 31.5MB/s]

Extracting files...
Geographic metadata located at: /root/.cache/kagglehub/datasets/saidmassinissafazez/snakes-species-details-dataset/versions/1
--2026-06-30 03:46:03--  https://ptak.felk.cvut.cz/plants/plants/SnakeCLEF2023/SnakeCLEF2023-train-medium_size.tar.gz
Resolving ptak.felk.cvut.cz (ptak.felk.cvut.cz)... 

147.32.84.14
Connecting to ptak.felk.cvut.cz (ptak.felk.cvut.cz)|147.32.84.14|:443... connected.
  Self-signed certificate encountered.
	requested host name ‘ptak.felk.cvut.cz’.
HTTP request sent, awaiting response... 200 OK
Length: 18582130206 (17G) [application/x-gzip]
Saving to: ‘SnakeCLEF2023-train-medium_size.tar.gz’

SnakeCLEF2023-train 100%[===================>]  17.31G  23.9MB/s    in 12m 37s 

2026-06-30 03:58:41 (23.4 MB/s) - ‘SnakeCLEF2023-train-medium_size.tar.gz’ saved [18582130206/18582130206]


--- Scanning Tar Index for Target Filtering ---
  Bitis arietans              : Selected 497 items.
  Echis ocellatus             : Selected 16 items.
  Bitis gabonica              : Selected 108 items.
  Dendroaspis polylepis       : Selected 205 items.
  Naja nigricollis            : Selected 123 items.
  Dispholidus typus           : Selected 299 items.
Extracting isolated target files from archive...
Target extraction complete.

--- Flattening and Cleaning Directory Layouts --

Copying files: 1248 files [00:00, 2963.37 files/s]



--- Constructing Partial Fine-Tuning Architectural Parameters ---
Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to /root/.cache/torch/hub/checkpoints/efficientnet_b0_rwightman-7f5810bc.pth


100%|██████████| 20.5M/20.5M [00:00<00:00, 104MB/s]



--- Launching Training Loop ---
Epoch 1/10 -> Train Loss: 1.3213 | Train Acc: 0.5166 || Val Loss: 0.9936 | Val Acc: 0.6576
Epoch 2/10 -> Train Loss: 0.9782 | Train Acc: 0.6475 || Val Loss: 0.7955 | Val Acc: 0.7446
Epoch 3/10 -> Train Loss: 0.8703 | Train Acc: 0.6797 || Val Loss: 0.7278 | Val Acc: 0.7337
Epoch 4/10 -> Train Loss: 0.7809 | Train Acc: 0.6992 || Val Loss: 0.6904 | Val Acc: 0.7391
Epoch 5/10 -> Train Loss: 0.7436 | Train Acc: 0.7130 || Val Loss: 0.6812 | Val Acc: 0.7500
Epoch 6/10 -> Train Loss: 0.6946 | Train Acc: 0.7405 || Val Loss: 0.6309 | Val Acc: 0.7554
Epoch 7/10 -> Train Loss: 0.6356 | Train Acc: 0.7566 || Val Loss: 0.6053 | Val Acc: 0.7609
Epoch 8/10 -> Train Loss: 0.6129 | Train Acc: 0.7738 || Val Loss: 0.5927 | Val Acc: 0.7663
Epoch 9/10 -> Train Loss: 0.5763 | Train Acc: 0.7945 || Val Loss: 0.5999 | Val Acc: 0.7717
Epoch 10/10 -> Train Loss: 0.5985 | Train Acc: 0.7807 || Val Loss: 0.5741 | Val Acc: 0.8043

Model optimized. Checkpoint tracking saved to -> 'afric

In [7]:
# --- STEP 8: THE TRAINING LOOP ---
print("\n--- Launching Training Loop ---")
EPOCHS = 57
best_val_loss = float("inf")

for epoch in range(EPOCHS):
    model.train()
    running_loss, correct_train, total_train = 0.0, 0, 0

    for inputs, labels in train_loader:
        inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()

        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * inputs.size(0)
        _, preds = torch.max(outputs, 1)
        correct_train += torch.sum(preds == labels.data)
        total_train += inputs.size(0)

    epoch_train_loss = running_loss / total_train
    epoch_train_acc = correct_train.double() / total_train

    # Validation phase
    model.eval()
    val_loss_accum, correct_val, total_val = 0.0, 0, 0
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            outputs = model(inputs)
            loss = criterion(outputs, labels)

            val_loss_accum += loss.item() * inputs.size(0)
            _, preds = torch.max(outputs, 1)
            correct_val += torch.sum(preds == labels.data)
            total_val += inputs.size(0)

    epoch_val_loss = val_loss_accum / total_val
    epoch_val_acc = correct_val.double() / total_val

    print(f"Epoch {epoch+1}/{EPOCHS} -> "
          f"Train Loss: {epoch_train_loss:.4f} | Train Acc: {epoch_train_acc:.4f} || "
          f"Val Loss: {epoch_val_loss:.4f} | Val Acc: {epoch_val_acc:.4f}")

    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        torch.save(model.state_dict(), MODEL_SAVE_PATH)

print(f"\nModel optimized. Checkpoint tracking saved to -> '{MODEL_SAVE_PATH}'")

# --- STEP 9: WRAPPER CLINICAL INFERENCE FUNCTION ---
def execute_clinical_inference(image_path):
    """
    Evaluates an image path and attaches herpetological metadata to the result.
    """
    model.eval()
    img = Image.open(image_path).convert("RGB")
    tensor = data_transforms["eval"](img).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        logits = model(tensor)
        probs = F.softmax(logits, dim=1)
        conf, class_idx = torch.max(probs, 1)

    predicted_folder_name = CLASS_NAMES[class_idx.item()]
    clinical_meta = VENOM_REFERENCE.get(predicted_folder_name, {
        "common_name": "Unknown", "fang_type": "N/A", "venom_risk": "N/A", "venom_action": "N/A", "clinical_note": "No match found."
    })

    return {
        "status": "Inference Executed",
        "species_scientific": predicted_folder_name.replace("_", " "),
        "confidence": round(conf.item(), 4),
        "clinical_data": clinical_meta
    }

# Example validation invocation syntax:
# print(execute_clinical_inference("path_to_sample_test_snake.jpg"))


--- Launching Training Loop ---
Epoch 1/57 -> Train Loss: 0.2014 | Train Acc: 0.9242 || Val Loss: 0.6481 | Val Acc: 0.8207
Epoch 2/57 -> Train Loss: 0.2401 | Train Acc: 0.9173 || Val Loss: 0.7221 | Val Acc: 0.8098
Epoch 3/57 -> Train Loss: 0.2045 | Train Acc: 0.9346 || Val Loss: 0.6963 | Val Acc: 0.8370
Epoch 4/57 -> Train Loss: 0.1705 | Train Acc: 0.9426 || Val Loss: 0.7010 | Val Acc: 0.8315
Epoch 5/57 -> Train Loss: 0.1500 | Train Acc: 0.9483 || Val Loss: 0.7152 | Val Acc: 0.8152
Epoch 6/57 -> Train Loss: 0.2064 | Train Acc: 0.9311 || Val Loss: 0.7315 | Val Acc: 0.8152
Epoch 7/57 -> Train Loss: 0.1553 | Train Acc: 0.9552 || Val Loss: 0.8045 | Val Acc: 0.8207
Epoch 8/57 -> Train Loss: 0.1615 | Train Acc: 0.9472 || Val Loss: 0.7006 | Val Acc: 0.8533
Epoch 9/57 -> Train Loss: 0.1890 | Train Acc: 0.9414 || Val Loss: 0.7402 | Val Acc: 0.8424
Epoch 10/57 -> Train Loss: 0.1906 | Train Acc: 0.9311 || Val Loss: 0.7932 | Val Acc: 0.8098
Epoch 11/57 -> Train Loss: 0.1496 | Train Acc: 0.9529 ||

In [13]:
print(f"\n--- Loading the saved model from '{MODEL_SAVE_PATH}' ---")

# Instantiate a fresh model with the same architecture
loaded_model = models.efficientnet_b0(weights=None) # No pretrained weights for loading state_dict
num_features = loaded_model.classifier[1].in_features
loaded_model.classifier[1] = nn.Linear(num_features, len(CLASS_NAMES))

# Load the state dictionary
loaded_model.load_state_dict(torch.load(MODEL_SAVE_PATH))
loaded_model = loaded_model.to(DEVICE)
loaded_model.eval() # Set the model to evaluation mode

print("Model loaded successfully!")
print("Example of loaded model's state_dict keys (first 5):")
for i, key in enumerate(list(loaded_model.state_dict().keys())):
    if i >= 5: # Print only first 5 for brevity
        break
    print(key)


--- Loading the saved model from 'african_snake_efficientnet.pth' ---
Model loaded successfully!
Example of loaded model's state_dict keys (first 5):
features.0.0.weight
features.0.1.weight
features.0.1.bias
features.0.1.running_mean
features.0.1.running_var


In [14]:
from google.colab import files

files.download(MODEL_SAVE_PATH)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>